# YOLO26-P2 LLM-GE commands

This notebook has one fresh-run path: a YOLO26-P2 search with up to four eight-hour LLM server windows (32 hours total). The chain stops early if generation 20 completes. Slurm submits separate eight-hour jobs; there is no single 32-hour server job.

Follow sections 1–4 in order and in the same PACE shell after connecting. Do not submit the server or controller twice for one tag. Sections 5–7 cover monitoring and results. Finish any older search that needs its original checkpoint fingerprint before pulling code or regenerating Slurm scripts in this checkout.


## 1. Connect from the Mac

```bash
ssh agupta3057@login-ice.pace.gatech.edu
```

## 2. Enter the repository on PACE

```bash
cd /storage/ice1/6/9/agupta3057/llm-guided-evolution-object-detection
module load uv
```

## 3. Prepare a fresh 32-hour YOLO26-P2 chain

Use this only for a **new tag**, after any older tag that needs the original checkpoint fingerprint has finished. After sections 1–2, paste this block on PACE. Stop if `git pull`, a data check, the configuration check, or `slurm.py` fails. Do not start another LLM-GE run in this shared checkout while the chain is active.

```bash
prepare_fresh_yolo26_chain() {
  git pull --ff-only origin fred-custom-seeds || return 1
  export FRED_SEED_FAMILY=yolo26_p2 FRED_NUM_GENERATIONS=21
  export LLMGE_PORT=8479 LLMGE_CLUSTER=pace-ice
  export LOCAL=false LOCAL_LLM=true INFERENCE_SUBMISSION=false
  export LLMGE_LLM_MODEL=mixtral SUBMIT_ISLAND_CONTROLLER=0
  export HF_HUB_DISABLE_XET=1
  unset FRED_EVENTS LLMGE_BOUNDED_SESSION LLMGE_MODEL_PATH
  unset LLM_MAX_NEW_TOKENS LLMGE_EVAL_NO_PROGRESS_TIMEOUT_SECONDS
  unset LLMGE_MIN_SERVER_REMAINING_SECONDS LLMGE_REAL_SQUEUE
  export FRED_RUN_TAG="yolo26_p2_$(date +%Y%m%d)_eot0_trial$(date +%H%M%S)"
  test ! -e "fred_runs/$FRED_RUN_TAG" || { echo 'Tag already exists; stop'; return 1; }
  test -s yolo26_chain_continue.sbatch || { echo 'Chain script missing'; return 1; }
  test -s sota/FRED_LLM_GE/phase1_detection/seeds/yolo11/data_cache/challenging/search_train_step50/data.yaml || { echo 'Training data missing'; return 1; }
  test -s sota/FRED_LLM_GE/phase1_detection/seeds/yolo11/data_cache/challenging/search_val_step10/data.yaml || { echo 'Validation data missing'; return 1; }
  uv run python -c 'from src.cfg import constants as c; assert c.SEED_FAMILY == "yolo26_p2" and c.num_generations == 21 and str(c.SEED_NETWORK).endswith("seed_yolo26n_p2.yaml"); print(c.SEED_FAMILY, c.RUN_TAG, c.SEED_NETWORK, c.num_generations)' || return 1
  mkdir -p run_job_outputs/{server,islands,evolution,evaluation} || return 1
  uv run python slurm.py || return 1
  test -x .venv/bin/python || { echo 'Python environment missing'; return 1; }
  grep -q 'Recorded server end time' server.sh || { echo 'Server deadline fix missing'; return 1; }
  printf '%s\n' "$FRED_RUN_TAG" > run_job_outputs/yolo26_current_tag.txt || return 1
  echo "fresh chain ready: $FRED_RUN_TAG"
  git log -1 --oneline
}
prepare_fresh_yolo26_chain
```

The configuration should print `yolo26_p2`, the new tag, `seed_yolo26n_p2.yaml`, and `21`. Record the tag and current Git commit. The chain keeps that same checkout and generated scripts across windows. The current-tag file is required at each automatic handoff.


## 4. Submit the full 32-hour chain

Paste this one block in the same PACE shell immediately after section 3. It submits the first eight-hour server, one controller, and a transition that schedules three more eight-hour windows. The controller waits for server readiness; no live log viewing or manual resubmission is needed.

```bash
export FRED_CHAIN_EXTRA_WINDOWS=3
export FRED_CHAIN_ROOT="$PWD/fred_runs/$FRED_RUN_TAG/chain"
mkdir -p "$FRED_CHAIN_ROOT"
SERVER_SUBMITTED=0
if [[ $FRED_CHAIN_EXTRA_WINDOWS =~ ^[1-9][0-9]*$ ]] && \
   test -s yolo26_chain_continue.sbatch && \
   test -s server.sh && test -s run.sh && \
   test -x .venv/bin/python && \
   test ! -e "$FRED_CHAIN_ROOT/jobs.tsv" && \
   test "$(cat run_job_outputs/yolo26_current_tag.txt)" = "$FRED_RUN_TAG" && \
   SERVER_JOB_ID=$(sbatch --parsable --partition=coe-gpu --export=ALL server.sh); then
  export LLMGE_SERVER_JOB_ID=${SERVER_JOB_ID%%;*}
  printf '%s\n' "$LLMGE_SERVER_JOB_ID" > run_job_outputs/yolo26_current_server.id
  echo "server job: $LLMGE_SERVER_JOB_ID"
  SERVER_SUBMITTED=1
else
  echo 'Preflight or server submission failed. No 32-hour chain was queued.'
fi
if test "$SERVER_SUBMITTED" = 1 && \
   RUN_JOB_ID=$(sbatch --parsable --export=ALL run.sh); then
  RUN_JOB_ID=${RUN_JOB_ID%%;*}
  printf '%s\n' "$RUN_JOB_ID" > run_job_outputs/yolo26_current_controller.id
  printf 'server\t%s\tfirst\ncontroller\t%s\t%s\n' \
    "$LLMGE_SERVER_JOB_ID" "$RUN_JOB_ID" "$LLMGE_SERVER_JOB_ID" > "$FRED_CHAIN_ROOT/jobs.tsv"
  echo "controller job: $RUN_JOB_ID"
  if CHAIN_JOB_ID=$(sbatch --parsable \
       --dependency="afterok:${RUN_JOB_ID},afterany:${LLMGE_SERVER_JOB_ID}" \
       --export=ALL --output="$FRED_CHAIN_ROOT/transition-%j.out" \
       yolo26_chain_continue.sbatch "$RUN_JOB_ID" "$LLMGE_SERVER_JOB_ID" \
       "$FRED_CHAIN_EXTRA_WINDOWS"); then
    CHAIN_JOB_ID=${CHAIN_JOB_ID%%;*}
    printf 'transition\t%s\t%s\n' "$CHAIN_JOB_ID" "$RUN_JOB_ID" >> "$FRED_CHAIN_ROOT/jobs.tsv"
    echo "automatic continuation queued: $CHAIN_JOB_ID"
    squeue -j "$RUN_JOB_ID,$CHAIN_JOB_ID" -o '%i %T %j %R %L'
  else
    echo 'Controller submitted, but continuation was not queued. Investigate before the server window ends.'
  fi
else
  echo 'Controller submission or preflight failed. No continuation was queued.'
fi
```

Look for all three IDs: server job, controller job, and automatic continuation queued. A transition shown as PENDING (Dependency) is expected. The transition checks the saved status and checkpoint pair before each handoff. If generation 20 completes, no further windows are submitted. Do not run this block twice for the same tag.


## 5. Check progress and errors during the chain

Run this in a new PACE shell whenever you want an update. The current-ID files change at each handoff. The first transition stays pending until the controller exits cleanly and its server allocation ends.

```bash
cd /storage/ice1/6/9/agupta3057/llm-guided-evolution-object-detection
export FRED_RUN_TAG="$(cat run_job_outputs/yolo26_current_tag.txt)"
FRED_CHAIN_ROOT="$PWD/fred_runs/$FRED_RUN_TAG/chain"
SERVER_JOB_ID="$(cat run_job_outputs/yolo26_current_server.id)"
RUN_JOB_ID="$(cat run_job_outputs/yolo26_current_controller.id)"
TRANSITION_JOB_ID="$(awk -F '	' '$1=="transition" {id=$2} END {print id}' "$FRED_CHAIN_ROOT/jobs.tsv")"
JOB_IDS="$(awk -F '	' 'NF>=2 {printf "%s%s", sep, $2; sep=","}' "$FRED_CHAIN_ROOT/jobs.tsv")"
cat "$FRED_CHAIN_ROOT/jobs.tsv"
squeue -j "$JOB_IDS" -o '%i %T %j %R %L'
sacct -j "$RUN_JOB_ID" --format=JobID,State,ExitCode,Elapsed -P
if test -s "fred_runs/$FRED_RUN_TAG/status.json"; then
  cat "fred_runs/$FRED_RUN_TAG/status.json"
else
  echo 'Run status has not been written yet.'
fi
SERVER_LOG="run_job_outputs/server/slurm-$SERVER_JOB_ID.out"
CONTROLLER_LOG="run_job_outputs/islands/slurm-$RUN_JOB_ID.out"
if test -f "$SERVER_LOG"; then tail -n 50 "$SERVER_LOG"; else echo 'Server log pending.'; fi
if test -f "$CONTROLLER_LOG"; then tail -n 80 "$CONTROLLER_LOG"; else echo 'Controller log pending.'; fi
if test -n "$TRANSITION_JOB_ID"; then
  sacct -j "$TRANSITION_JOB_ID" --format=JobID,State,ExitCode,Elapsed -P
  if test -f "$FRED_CHAIN_ROOT/transition-$TRANSITION_JOB_ID.out"; then
    tail -n 80 "$FRED_CHAIN_ROOT/transition-$TRANSITION_JOB_ID.out"
  fi
fi
```

Show recent generation milestones and potential errors in the current controller log:

```bash
if test -f "$CONTROLLER_LOG"; then
  grep -nE 'Finished Generation|Generation [0-9]+ failures:|Traceback|RuntimeError|ERROR' "$CONTROLLER_LOG" | tail -n 40
fi
```

A line showing zero generation failures is normal. A nonzero count needs candidate-level inspection. A controller or transition marked FAILED, a traceback, or DependencyNeverSatisfied needs investigation before any manual submission. A PENDING controller can simply be waiting for Slurm scheduling.


## 6. Verify completion and create results and Pareto plots

Do this after the latest controller finishes. A complete 21-generation run requires the latest controller to show COMPLETED with exit code 0:0, status complete, and generation 20 saved. In a new PACE shell:

```bash
cd /storage/ice1/6/9/agupta3057/llm-guided-evolution-object-detection
module load uv
export FRED_SEED_FAMILY=yolo26_p2
export FRED_RUN_TAG="$(cat run_job_outputs/yolo26_current_tag.txt)"
RUN_JOB_ID="$(cat run_job_outputs/yolo26_current_controller.id)"
sacct -j "$RUN_JOB_ID" --format=JobID,State,ExitCode,Elapsed -P
.venv/bin/python -c 'import json,sys; s=json.load(open(sys.argv[1])); assert s.get("status") == "complete" and s.get("last_completed_generation") == 20, s; print(s)' "fred_runs/$FRED_RUN_TAG/status.json"
test -s "fred_runs/$FRED_RUN_TAG/checkpoints/checkpoint_gen_20.pkl" && echo 'generation 20 checkpoint ready'
test -s "fred_runs/$FRED_RUN_TAG/checkpoints/global_gen_20.pkl" && echo 'generation 20 global state ready'
uv run python analysis/scripts/fred_pareto_graphs.py
ls -lh "analysis/results/analysis/fred_pareto/$FRED_RUN_TAG"
head -n 10 "analysis/results/analysis/fred_pareto/$FRED_RUN_TAG/pareto_models.csv"
```

The output folder contains candidates.csv, pareto_models.csv, per-generation Pareto plots, and pareto2d_final.png. These are search-validation results. A plot from partial checkpoints is not a completed-run result.


## 7. Copy the plots, CSVs, and raw run data to the Mac

Run in a **Mac terminal** with the PACE VPN connected, after section 6 succeeds. This reads the current run tag from PACE. The raw run copy may be large.

```bash
PACE_ROOT=/storage/ice1/6/9/agupta3057/llm-guided-evolution-object-detection
FRED_RUN_TAG=$(ssh agupta3057@login-ice.pace.gatech.edu "cat $PACE_ROOT/run_job_outputs/yolo26_current_tag.txt")
LOCAL_ROOT="$HOME/Desktop/GP/llm-guided-evolution-object-detection/sota/FRED_LLM_GE/PACE-ICE Run Results/YOLO26-P2-LLM-GE-$FRED_RUN_TAG"
mkdir -p "$LOCAL_ROOT/pareto" "$LOCAL_ROOT/run"
rsync -av "agupta3057@login-ice.pace.gatech.edu:$PACE_ROOT/analysis/results/analysis/fred_pareto/$FRED_RUN_TAG/" "$LOCAL_ROOT/pareto/"
rsync -avP "agupta3057@login-ice.pace.gatech.edu:$PACE_ROOT/fred_runs/$FRED_RUN_TAG/" "$LOCAL_ROOT/run/"
```

The tagged run folder contains checkpoints, status, candidate files, and evaluation artifacts. Slurm logs are under the separate run_job_outputs folder on PACE.

## 8. Diagnose the November 12 controller failure before another submission

This applies to `yolo26_p2_20261112_eot0_trial080235`. Controller 6984090 failed after repeated `squeue` timeouts during generation-0 creation. The automatic transition 6984091 cannot run because it requires that controller to complete successfully. Server 6984011 reached its eight-hour limit. The generation-0 checkpoint pair in the log is **provisional**; it does not prove generation 0 was evaluated or that a Pareto result exists. Do not rerun section 4 for this tag, pull code into its PACE checkout, or regenerate its Slurm scripts.

In a PACE terminal, collect the current state and check for outstanding operator/evaluation jobs:

```bash
cd /storage/ice1/6/9/agupta3057/llm-guided-evolution-object-detection
TAG=yolo26_p2_20261112_eot0_trial080235
sacct -j 6984011,6984090,6984091 --format=JobID,JobName,State,ExitCode,Elapsed -P
squeue -u "$USER" -o '%i %T %j %R %L'
if test -s "fred_runs/$TAG/status.json"; then cat "fred_runs/$TAG/status.json"; else echo 'No run status yet'; fi
ls -lh "fred_runs/$TAG/checkpoints"/checkpoint_gen_*.pkl "fred_runs/$TAG/checkpoints"/global_gen_*.pkl
if test -f run_job_outputs/islands/slurm-6984090.out; then tail -n 100 run_job_outputs/islands/slurm-6984090.out; fi
```

Share that output before resuming this tag. Confirm the state of submitted generation-0 operator jobs first; the controller had already submitted at least two when it failed. If 6984091 still shows `DependencyNeverSatisfied`, it can be removed from the queue with `scancel 6984091`; that does not resume or cancel any operator job. Do not submit another controller while old operator jobs may still be running.

For a **fresh tag after this run is closed**, use sections 3–4 with the corrected code. The new `server.sh` records `SLURM_JOB_END_TIME` in `fred_runs/<tag>/server_windows/<server-job-id>.deadline`; the controller uses that saved deadline and HTTP readiness without repeated scheduler queries during operator submission. Confirm `Recorded server end time for job ...` appears in each server log. If it does not, stop and inspect the server environment before submitting a controller. This avoids the observed `squeue` timeout path; it cannot prevent all Slurm or model failures. See [Slurm's documented SLURM_JOB_END_TIME](https://slurm.schedmd.com/sbatch.html#SECTION_OUTPUT-ENVIRONMENT-VARIABLES).


## 9. Close the failed November 12 chain, then start a fresh 32-hour run

Use this only if you are **abandoning** `yolo26_p2_20261112_eot0_trial080235`. Its controller 6984090 failed, server 6984011 timed out, and transition 6984091 was blocked in the last screenshot. This block obtains the old operator IDs from that controller's log, cancels any that are still queued/running, cancels the blocked transition, and verifies those IDs left the queue. It does not touch unrelated Slurm jobs. If Slurm returns an error or the `old run jobs cleared` line does not print, stop.

```bash
cd /storage/ice1/6/9/agupta3057/llm-guided-evolution-object-detection
close_failed_yolo26_run() {
  local log=run_job_outputs/islands/slurm-6984090.out
  local worker_ids active_ids job_id
  test -s "$log" || { echo 'Old controller log missing; stop'; return 1; }
  worker_ids=$(grep -oE 'Submitted batch job [0-9]+' "$log" | awk '{print $4}' | sort -u | paste -sd, -)
  echo "Old operator job IDs: ${worker_ids:-none}"
  squeue -j 6984091 -o '%i %T %j %R' || return 1
  if test -n "$worker_ids"; then
    active_ids=$(squeue -h -j "$worker_ids" -o '%i') || return 1
    for job_id in $active_ids; do scancel "$job_id" || return 1; done
  fi
  if squeue -h -j 6984091 -o '%i' | grep -q '^6984091$'; then
    scancel 6984091 || return 1
  fi
  if test -n "$worker_ids"; then
    active_ids=$(squeue -h -j "$worker_ids" -o '%i') || return 1
    test -z "$active_ids" || { echo "Old operator jobs remain: $active_ids"; return 1; }
  fi
  active_ids=$(squeue -h -j 6984091 -o '%i') || return 1
  test -z "$active_ids" || { echo 'Old transition remains queued'; return 1; }
  echo 'old run jobs cleared'
}
close_failed_yolo26_run
```

Check `squeue -u "$USER" -o '%i %T %j %R'` for any other jobs belonging to the old tag. Once the old jobs are cleared, the old tag's partial checkpoint remains on disk. Follow **sections 3 and 4** in the same PACE shell to pull commit `8b3953257` or later, create a **new tag**, regenerate `server.sh`/`run.sh`, and submit the four-window chain. The old checkpoint cannot be resumed after pulling the new code because its protocol fingerprint includes the Git revision. Record the three new IDs and check section 5 after the server starts.
